# Análise de dados TCP-CI

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('./T CELL/DENV 2 - T Cell Prediction - Class I.csv')
df

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhcpan_el core,netmhcpan_el icore,netmhcpan_el score,netmhcpan_el percentile
0,1,VTRLENLMW,60,68,9,HLA-B*57:01,60,0.01,VTRLENLMW,VTRLENLMW,0.993804,0.01
1,1,ASGKLITEW,303,311,9,HLA-B*57:01,303,0.01,ASGKLITEW,ASGKLITEW,0.990493,0.01
2,1,ASGKLITEW,303,311,9,HLA-B*58:01,303,0.01,ASGKLITEW,ASGKLITEW,0.989009,0.01
3,1,ETAECPNTNR,139,148,10,HLA-A*68:01,483,0.01,ETAEPNTNR,ETAECPNTNR,0.981131,0.01
4,1,QPTELKYSW,107,115,9,HLA-B*53:01,107,0.01,QPTELKYSW,QPTELKYSW,0.980907,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...
36985,1,LKEKEENLVNS,338,348,11,HLA-A*32:01,1025,100.00,KEKENLVNS,KEKEENLVNS,0.000000,100.00
36986,1,LKEKEENLVNS,338,348,11,HLA-B*53:01,1025,100.00,LKEENLVNS,LKEKEENLVNS,0.000000,100.00
36987,1,EKEENLVNSLVT,340,351,12,HLA-A*11:01,1369,100.00,ENLVNSLVT,EKEENLVNSLVT,0.000000,100.00
36988,1,EKEENLVNSLVT,340,351,12,HLA-A*32:01,1369,100.00,EEENLVNSL,EKEENLVNSL,0.000000,100.00


## Selecionando Epítopos com median binding percentile menor que 5.

In [3]:
df_mbp_m5 = df[df['median binding percentile'] < 5].copy()
df_mbp_m5

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhcpan_el core,netmhcpan_el icore,netmhcpan_el score,netmhcpan_el percentile
0,1,VTRLENLMW,60,68,9,HLA-B*57:01,60,0.01,VTRLENLMW,VTRLENLMW,0.993804,0.01
1,1,ASGKLITEW,303,311,9,HLA-B*57:01,303,0.01,ASGKLITEW,ASGKLITEW,0.990493,0.01
2,1,ASGKLITEW,303,311,9,HLA-B*58:01,303,0.01,ASGKLITEW,ASGKLITEW,0.989009,0.01
3,1,ETAECPNTNR,139,148,10,HLA-A*68:01,483,0.01,ETAEPNTNR,ETAECPNTNR,0.981131,0.01
4,1,QPTELKYSW,107,115,9,HLA-B*53:01,107,0.01,QPTELKYSW,QPTELKYSW,0.980907,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...
2519,1,FIEVKNCHW,217,225,9,HLA-A*24:02,217,4.90,FIEVKNCHW,FIEVKNCHW,0.003306,4.90
2520,1,ESEMIIPKNLA,238,248,11,HLA-B*44:02,925,4.90,EEMIIPKNL,ESEMIIPKNL,0.002875,4.90
2521,1,MEIRPLKEKE,333,342,10,HLA-B*44:02,677,4.90,MEIRPLKKE,MEIRPLKEKE,0.002854,4.90
2522,1,FIEVKNCHW,217,225,9,HLA-B*44:02,217,4.90,FIEVKNCHW,FIEVKNCHW,0.002809,4.90


## Agrupando por pepitideos e agregando colunas pertinentes.

In [4]:
epitopos_repetidos = (
    df_mbp_m5
    .groupby('peptide', as_index=False)
    .agg(
        start=("start", "first"),
        end=("end", "first"),
        qte_de_alelos=("allele", "nunique"),
        median_binding_percentile=(
            "median binding percentile",
            "median"
        ),
        alelos=(
            "allele",
            lambda x: ", ".join(sorted(x.unique()))
        )
    ))

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,AAIKDNRAV,186,194,7,2.70,"HLA-A*02:03, HLA-A*02:06, HLA-A*68:02, HLA-B*0..."
1,AAIKDNRAVH,186,195,2,4.20,"HLA-A*30:02, HLA-B*15:01"
2,ADMGYWIESAL,196,206,1,3.00,HLA-B*40:01
3,AECPNTNRA,141,149,3,1.00,"HLA-B*40:01, HLA-B*44:02, HLA-B*44:03"
4,AECPNTNRAW,141,150,8,1.70,"HLA-A*01:01, HLA-A*23:01, HLA-B*40:01, HLA-B*4..."
...,...,...,...,...,...,...
585,YSWKTWGKA,113,121,2,4.40,"HLA-A*30:02, HLA-A*68:02"
586,YSWKTWGKAK,113,122,6,2.05,"HLA-A*03:01, HLA-A*11:01, HLA-A*30:01, HLA-A*3..."
587,YSWKTWGKAKM,113,123,2,2.40,"HLA-B*57:01, HLA-B*58:01"
588,YWIESALNDTW,200,210,9,1.10,"HLA-A*23:01, HLA-A*24:02, HLA-A*32:01, HLA-B*4..."


# Filtragem por epítopos presentes em mais de determinada quantidade de alelos.

In [5]:
epitopos_repetidos = epitopos_repetidos[
    epitopos_repetidos["qte_de_alelos"] >= 10
].reset_index(drop=True)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,ASGKLITEW,303,311,12,1.600,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
1,AVHADMGYW,193,201,10,1.645,"HLA-A*23:01, HLA-A*26:01, HLA-A*30:02, HLA-A*3..."
2,CHWPKSHTL,223,231,12,2.600,"HLA-A*23:01, HLA-A*24:02, HLA-A*30:02, HLA-A*3..."
3,CTLPPLRYR,316,324,11,0.960,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2..."
4,DTWKIEKASF,208,217,12,2.950,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
5,ELKYSWKTW,110,118,11,1.500,"HLA-A*23:01, HLA-A*24:02, HLA-A*26:01, HLA-A*3..."
6,FIEVKNCHW,217,225,10,3.200,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*3..."
7,FITDNVHTW,20,28,22,1.150,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."
8,FQPESPSKL,34,42,18,1.600,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."
9,FTTNIWLKL,163,171,16,1.650,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."


In [6]:
epitopos_repetidos = (
    epitopos_repetidos
    .sort_values(
        ["qte_de_alelos", "median_binding_percentile"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,FITDNVHTW,20,28,22,1.150,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."
1,TQITGPWHL,262,270,22,1.750,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*2..."
2,HTWTEQYKF,26,34,21,1.500,"HLA-A*01:01, HLA-A*02:06, HLA-A*11:01, HLA-A*2..."
3,SLRPQPTEL,103,111,21,2.500,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."
4,KLKEKQDVF,170,178,20,2.100,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."
5,RPQPTELKY,105,113,19,1.400,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2..."
6,STESHNQTF,125,133,18,1.200,"HLA-A*01:01, HLA-A*02:06, HLA-A*23:01, HLA-A*2..."
7,FQPESPSKL,34,42,18,1.600,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."
8,KTWGKAKML,116,124,18,2.400,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*0..."
9,SENEVKLTI,80,88,18,3.000,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:06, HLA-A*2..."
